# 3. Fine-tuning

Whisper learned its weights from 680,000 hours of audio. **Fine-tuning** continues that training on your recordings for a few minutes, with small steps, so the weights shift just enough to fit your voice and your words without forgetting everything else.

In this notebook you write the whole training pipeline yourself, in plain PyTorch:

1. **Part 1, one example.** Turn one recording into the two things training needs: input features and labels.
2. **Part 2, batches.** Write the **data collator**, which packs several examples into one batch of tensors, and look at exactly what the decoder reads during training.
3. **Part 3, one forward pass.** Run the model on a batch, get its loss, and compute that loss yourself.
4. **Part 4, a sanity check.** Train on a single batch until the model memorizes it, to prove your training step works.
5. **Part 5, the training loop.** Train on the whole training set for a few epochs, tracking validation WER after each one.
6. **Parts 6 and 7.** See what changed, and save the model.
7. **Part 8.** Move your functions into `whisper_ft.py` for notebook 04.

The test set stays locked away the whole time. Every decision here uses the **validation** set.

## The whole pipeline at a glance

Training repeats one loop. Each row below is one step, the code that does it, and the type of data it produces. Every part of this notebook fills in one or two rows, so come back to this table whenever you lose track.

| # | step | code | produces |
|---|---|---|---|
| 1 | read a recording | `read_wav(row["path"])` | `np.ndarray`, float32, shape `(n_samples,)` |
| 2 | audio to features | `processor.feature_extractor(...)` | `np.ndarray`, float32, shape `(80, 3000)` |
| 3 | text to labels | `processor.tokenizer(...)` | a list of int (token ids), with a different length for each sentence |
| 4 | examples to a batch | your `collate(...)` | a dict of tensors: `input_features`, float32, `(B, 80, 3000)`, and `labels`, int64, `(B, L)` |
| 5 | forward pass | `model(**batch)` | `.loss`, a tensor holding one number, and `.logits`, float32, `(B, L, 51864)` |
| 6 | backward pass | `loss.backward()` | a gradient on every weight, stored in the weight's `.grad`, with the weight's shape |
| 7 | update | `optimizer.step()` | every weight moves a small step |

`B` is the batch size (how many recordings go through together), `L` is the length of the longest label sequence in the batch, and 51,864 is the number of tokens in Whisper's vocabulary.

## New data types in this notebook

**`torch.Tensor`** is PyTorch's array. It works much like a NumPy array, with two extra abilities: it can live on a GPU, and it can remember how it was computed, so PyTorch can work out gradients automatically. The attributes you'll check most:

| attribute | meaning | example |
|---|---|---|
| `t.shape` | the size of each dimension | `torch.Size([8, 80, 3000])` |
| `t.dtype` | the number type | `torch.float32` for features and weights, `torch.int64` for token ids |
| `t.ndim` | the number of dimensions | `3` |
| `t.item()` | turns a one-number tensor into a Python `float` or `int` | `loss.item()` |
| `t.tolist()` | turns a tensor into (nested) Python lists | `labels[0].tolist()` |
| `t.requires_grad` | whether PyTorch tracks gradients for it | `True` for the weights you train |

To move between NumPy and PyTorch: `torch.from_numpy(array)` makes a tensor that shares memory with the array (fast, no copying), and `tensor.numpy()` goes the other way.

Indexing works like NumPy: `labels[0]` is the first row, `labels[:, :5]` is the first 5 columns of every row, and `labels[0, :5]` is the first 5 values of the first row.

**The model's output** is a `Seq2SeqLMOutput`: a dict-like object whose fields you read as attributes, mainly `outputs.loss` and `outputs.logits`.

**A `DataLoader`** (Part 5) is PyTorch's batching tool. It takes your list of examples, groups them into batches, reshuffles them every epoch, and calls your collator on each group.

## Setup

This cell loads your recordings and the split from notebook 02, and a fresh whisper-tiny.en.

- `rows_for_split(rows, splits, "train")` returns the recordings whose sentence is in the training set: a list of dicts, as in notebook 02.
- `torch.manual_seed(SEED)` fixes PyTorch's random numbers (used for shuffling), so a rerun gives the same results.
- `SOT`, `NO_TIMESTAMPS`, and `EOT` are the ids of three special tokens, defined in `whisper_ft.py`: `<|startoftranscript|>`, `<|notimestamps|>`, and `<|endoftext|>`.
- **`SMOKE_TEST`**: set it to `True` to run the whole notebook on 16 training and 8 validation recordings, for 1 epoch. It finishes in a few minutes, so it's a cheap way to find bugs before the real run, which takes about 15 minutes. Set it back to `False` for the real run.

In [ ]:
%load_ext autoreload
%autoreload 2

import copy
import inspect
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from torch.utils.data import DataLoader
from transformers import WhisperForConditionalGeneration
from transformers.models.whisper.modeling_whisper import shift_tokens_right

from whisper_ft import (
    EOT,
    MODEL_NAME,
    MODELS_DIR,
    NO_TIMESTAMPS,
    RESULTS_DIR,
    SAMPLE_RATE,
    SOT,
    corpus_wer,
    load_metadata,
    load_splits,
    processor,
    read_wav,
    rows_for_split,
    save_json,
    transcribe,
)

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
SMOKE_TEST = False
SEED = 0
torch.manual_seed(SEED)

rows = load_metadata()
splits = load_splits()
train_rows = rows_for_split(rows, splits, "train")
val_rows = rows_for_split(rows, splits, "val")
if SMOKE_TEST:
    train_rows, val_rows = train_rows[:16], val_rows[:8]

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

print(f"{len(train_rows)} training recordings, {len(val_rows)} validation recordings")
print(f"Special tokens: <|startoftranscript|> = {SOT}, <|notimestamps|> = {NO_TIMESTAMPS}, <|endoftext|> = {EOT}")

## Part 1: from one recording to one training example

### Inputs and targets

A training example is a pair: what the model sees, and the answer it should produce.

- **`input_features`**: the log-mel spectrogram of the recording, exactly as in notebook 01. A NumPy array, float32, shape `(80, 3000)`.
- **`labels`**: the correct transcript as token ids, the same ids `generate()` would produce if the model were perfect. A list of int.

The labels for " Move the model." look like this:

    <|startoftranscript|>  <|notimestamps|>  ĠMove  Ġthe  Ġmodel   .   <|endoftext|>
           50257                50362        10028   262    2746   13       50256

They begin with the same prompt `generate()` starts from, and end with `<|endoftext|>`. The model has to learn to produce that end token, or it would never know when to stop.

### Why a space goes in front of the text

Look at the tokens above: `ĠMove`, with a `Ġ`. Whisper was trained on transcripts that start with a space, so every word, including the first one, is a token with a leading space, and its own output starts that way too (in notebook 01, `batch_decode` gave you `" Mr. Quilter..."`). But tokenize `"Move the model."` without the space, and the first word becomes `Move`, without `Ġ`: a different token, which Whisper almost never produces right after the prompt. The next cell shows the difference.

Training on labels without the space would spend the start of fine-tuning on a formatting habit instead of your words. I measured it on 16 sentences: the loss on the first word was 9.4 without the space and 0.2 with it. So tokenize `" " + text`.

In [ ]:
for text in ["Move the model.", " Move the model."]:
    ids = processor.tokenizer(text).input_ids
    print(f"{text!r:20} -> {ids}")
    print(f"{'':20}    {processor.tokenizer.convert_ids_to_tokens(ids)}")

### Functions you'll use

- **`read_wav(path)`** (from `whisper_ft.py`) reads a WAV file and returns the waveform: a 1-D float32 NumPy array.
- **`processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="np")`** turns a waveform into a log-mel spectrogram, as in notebook 01. With `return_tensors="np"` it returns NumPy arrays instead of PyTorch tensors: examples stay NumPy arrays, and Part 2 turns whole batches into tensors. The result is a `BatchFeature` (dict-like), and its `.input_features` has shape `(1, 80, 3000)`, because the extractor always adds a batch dimension, even for one clip. Index it with `[0]` to get the `(80, 3000)` array.
- **`processor.tokenizer(text)`** returns a `BatchEncoding` (dict-like). Its `.input_ids` is the list of token ids (a `list` of `int`), with the prompt and `<|endoftext|>` already added.

### TODO(you): write `prepare_example(row)`

**Input:** `row`, one dict from `load_metadata()`. You need `row["path"]` and `row["text"]`.

**Output:** a dict with two keys:
- `"input_features"`: NumPy array, float32, shape `(80, 3000)`.
- `"labels"`: list of int, from tokenizing `" " + row["text"]`.

In [ ]:
# TODO(you): turn one recording into {"input_features": (80, 3000) array, "labels": list of token ids}.
def prepare_example(row):
    ...

In [ ]:
example = prepare_example(train_rows[0])
assert isinstance(example, dict) and set(example) == {"input_features", "labels"}, \
    "return a dict with the keys 'input_features' and 'labels'"
features, labels = example["input_features"], example["labels"]

assert isinstance(features, np.ndarray), f"input_features should be a NumPy array, not {type(features).__name__}"
assert features.shape == (80, 3000), f"expected shape (80, 3000), got {features.shape}; did you take [0]?"
assert features.dtype == np.float32
assert isinstance(labels, list) and all(isinstance(i, int) for i in labels), "labels should be a list of int"
assert NO_TIMESTAMPS in labels[:2] and labels[-1] == EOT, "labels should start with the prompt and end with <|endoftext|>"
first_word = processor.tokenizer.convert_ids_to_tokens(labels[labels.index(NO_TIMESTAMPS) + 1])
assert first_word.startswith("Ġ"), f"the first word token is {first_word!r}; it should start with Ġ, so tokenize ' ' + text"
assert processor.tokenizer.decode(labels, skip_special_tokens=True).strip() == train_rows[0]["text"]

print(train_rows[0]["text"])
print(processor.tokenizer.convert_ids_to_tokens(labels))
print("Correct.")

Now prepare every training example once and keep them in a list. Each feature array takes 80 × 3,000 × 4 bytes, about 1 MB, so a couple of hundred fit easily in memory, and you don't recompute features every epoch. `array.nbytes` is an array's size in bytes, and `time.perf_counter()` reads a clock, for timing.

Notice that the label lengths differ. That's the problem Part 2 solves.

In [ ]:
start = time.perf_counter()
train_examples = [prepare_example(row) for row in train_rows]
megabytes = sum(ex["input_features"].nbytes for ex in train_examples) / 1e6
print(f"Prepared {len(train_examples)} examples in {time.perf_counter() - start:.0f} s ({megabytes:.0f} MB of features)")

lengths = [len(ex["labels"]) for ex in train_examples]
print(f"Label lengths: shortest {min(lengths)}, longest {max(lengths)} tokens")

## Part 2: batches and the data collator

### Why batches

Training on one example at a time is slow and noisy. A **batch** sends several examples through the model at once: the computer processes them in parallel, and the update averages over all of them, so each step points in a more reliable direction. We'll use batches of 8.

### The problem: labels have different lengths

To stack examples into one tensor, every row needs the same length. The features already match, because Whisper always pads audio to 30 seconds, so every example is `(80, 3000)`, and stacking 8 gives `(8, 80, 3000)`.

The labels don't match: "Sounds good, see you at six." is about 12 tokens, and a long sentence might be 25. So you **pad** the shorter ones to the length of the longest one in the batch:

    labels:          ĠSounds  Ġgood  ...  .  <|endoftext|>
    after padding:   ĠSounds  Ġgood  ...  .  <|endoftext|>  -100  -100  -100

### Why -100

The loss function compares the model's prediction at every position with the label at that position. Padding positions aren't real text, so they must not count. PyTorch's cross-entropy loss skips every position whose label is **-100** (its `ignore_index`), so that's what you pad with. Hugging Face models follow the same rule.

There's a trap. Whisper's tokenizer pads with its padding token, and for Whisper, the padding token **is** `<|endoftext|>` (id 50256). So you can't find the padding by looking for 50256: that would also erase the real `<|endoftext|>` at the end of every sentence, and a model that never sees the end token in its labels never learns to stop. Use the **attention mask** that the tokenizer returns along with the padded ids instead: it's 1 for a real token and 0 for padding.

### Functions you'll use

- **`np.stack(list_of_arrays)`** stacks arrays of the same shape along a new first dimension: 8 arrays of shape `(80, 3000)` become one array of shape `(8, 80, 3000)`.
- **`torch.from_numpy(array)`** turns that array into a tensor with the same dtype (float32).
- **`processor.tokenizer.pad(list_of_dicts, return_tensors="pt")`** pads token ids. Give it a list of dicts, each with an `"input_ids"` key holding one example's labels: `[{"input_ids": ex["labels"]} for ex in batch]`. It returns a `BatchEncoding` (dict-like) with two tensors, both int64 with shape `(B, L)`: `"input_ids"`, padded on the right with 50256, and `"attention_mask"`, 1 for real tokens and 0 for padding.
- **`tensor.masked_fill(mask, value)`** returns a copy of `tensor` with `value` wherever `mask` is `True`. `mask` is a tensor of `True`/`False` values with the same shape; `attention_mask == 0` makes one.

The next cell runs these on two tiny made-up label lists, so you can see each step.

In [ ]:
demo = processor.tokenizer.pad([{"input_ids": [1, 2, 3]}, {"input_ids": [4, 5]}], return_tensors="pt")
print("input_ids:\n", demo["input_ids"])
print("attention_mask:\n", demo["attention_mask"])
print("attention_mask == 0:\n", demo["attention_mask"] == 0)
print("input_ids.masked_fill(attention_mask == 0, -100):\n", demo["input_ids"].masked_fill(demo["attention_mask"] == 0, -100))

### TODO(you): write `collate(batch)`

**Input:** `batch`, a list of examples from `prepare_example` (dicts with `"input_features"` and `"labels"`).

**Output:** a dict with two tensors:
- `"input_features"`: float32, shape `(B, 80, 3000)`.
- `"labels"`: int64, shape `(B, L)`, with -100 at every padding position, and the real tokens (including each sentence's `<|endoftext|>`) unchanged.

**Steps:**
1. Stack the features with `np.stack` and convert them with `torch.from_numpy`.
2. Pad the labels with `processor.tokenizer.pad`.
3. Replace the padding positions with -100, using the attention mask and `masked_fill`.
4. Return the dict. The keys must be exactly `"input_features"` and `"labels"`: in Part 3 you'll pass this dict straight to the model as keyword arguments, and those are the argument names it expects.

In [ ]:
# TODO(you): turn a list of examples into {"input_features": (B, 80, 3000) float32, "labels": (B, L) int64, -100 at padding}.
def collate(batch):
    ...

The check collates the shortest, a middle, and the longest training example, so there's real padding to check. `sorted(..., key=lambda ex: len(ex["labels"]))` sorts the examples by label length.

In [ ]:
by_length = sorted(train_examples, key=lambda ex: len(ex["labels"]))
trio = [by_length[0], by_length[len(by_length) // 2], by_length[-1]]
batch = collate(trio)

assert isinstance(batch, dict) and set(batch) == {"input_features", "labels"}, "return a dict with the keys 'input_features' and 'labels'"
assert isinstance(batch["input_features"], torch.Tensor) and isinstance(batch["labels"], torch.Tensor), "both values should be tensors"
assert batch["input_features"].shape == (3, 80, 3000) and batch["input_features"].dtype == torch.float32
labels = batch["labels"]
assert labels.dtype == torch.int64 and labels.ndim == 2 and labels.shape[0] == 3
for row, ex in zip(labels.tolist(), trio):
    real = [t for t in row if t != -100]
    assert row[len(real):] == [-100] * (len(row) - len(real)), "padding should be -100, on the right"
    assert real[-1] == EOT, "every row should keep its <|endoftext|>"
    assert NO_TIMESTAMPS in real[:2], "every row should start with the prompt tokens"
    assert processor.tokenizer.decode(real, skip_special_tokens=True) == processor.tokenizer.decode(ex["labels"], skip_special_tokens=True), \
        "the real tokens should be the example's labels, unchanged"
print(labels)
print("Correct.")

### What the decoder actually reads during training

In notebook 01, the decoder generated one token at a time, reading its *own* previous outputs. Training works differently: the decoder reads the **correct** transcript as its input and, at every position at once, predicts the next token. This is called **teacher forcing**, like a teacher who corrects you after every word, so one mistake doesn't derail the rest of the sentence. It's also why training is fast: all positions are computed in a single forward pass, instead of one pass per token.

To predict the next token, the decoder's input at each position has to be the label one step earlier. So the model builds its decoder input from your labels by **shifting them one position to the right**, and putting a start token in the first position:

    labels:              A      B   C   <|endoftext|>
    decoder input:   <start>    A   B        C

    position 0 reads <start> and must predict A; position 1 reads A and must predict B; and so on.

You don't do this shift yourself. When you call `model(input_features=..., labels=...)`, the model does it internally with a function called `shift_tokens_right`, using its configured start token, `model.config.decoder_start_token_id` (the same id as `SOT`). It also replaces any -100 in the decoder input with the padding id, since -100 isn't a real token.

The next cell calls that same function on two of your collated examples and prints the first tokens the decoder will read. Compare them with the prompt `generate()` starts from at inference time: `<|startoftranscript|><|notimestamps|>`.

In [ ]:
batch = collate(train_examples[:2])
decoder_input_ids = shift_tokens_right(batch["labels"], model.config.pad_token_id, model.config.decoder_start_token_id)

print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("labels, first 5:        ", processor.tokenizer.convert_ids_to_tokens(batch["labels"][0, :5].tolist()))
print("decoder input, first 5: ", processor.tokenizer.convert_ids_to_tokens(decoder_input_ids[0, :5].tolist()))
print("prompt at inference:    ", ["<|startoftranscript|>", "<|notimestamps|>"])

### Check your understanding

Do the first tokens the decoder reads during training match the prompt it reads at inference? If not, what would the model learn during training that it will never see when you use it? Fix your code if needed, rerun the collator's check cell and the cell above, and then run the check below.

In [ ]:
batch = collate(train_examples[:4])
decoder_input_ids = shift_tokens_right(batch["labels"], model.config.pad_token_id, model.config.decoder_start_token_id)
assert (decoder_input_ids[:, :2] == torch.tensor([SOT, NO_TIMESTAMPS])).all(), (
    "During training, the decoder should start by reading exactly the prompt it reads at inference, "
    "<|startoftranscript|><|notimestamps|>. Compare it with the decoder input printed above."
)
print("Correct: training and inference now start from the same prompt.")

## Part 3: one forward pass, and the loss

### What `model(input_features=..., labels=...)` does

1. The **encoder** turns each spectrogram into 1,500 vectors, as in notebook 01.
2. The model builds the decoder input by shifting the labels (Part 2).
3. The **decoder** reads it and outputs **logits**: at every position, one score for each of the 51,864 tokens in the vocabulary, so the shape is `(B, L, 51864)`.
4. It computes the **cross-entropy loss** between the logits and the labels, skipping the positions labeled -100, and returns both.

### Cross-entropy, step by step

At one position, softmax turns the 51,864 scores into probabilities that add up to 1. Say the correct next token is `ĠLoRA`, and the model gives it probability 0.9. The loss at that position is −log(0.9) ≈ 0.11. If it had given 0.01, the loss would be −log(0.01) ≈ 4.6. Confident and right costs almost nothing; confident and wrong costs a lot. The final loss is the average over all real (not -100) positions in the batch.

Two reference points:
- A model guessing at random gives every token probability 1/51,864, so its loss is log(51,864) ≈ **10.9**.
- `exp(loss)` is called **perplexity**: roughly, how many tokens the model is torn between at each step. A loss of 0.7 is a perplexity of about 2.

### Functions you'll use

- **`model(**batch)`**: the `**` unpacks a dict into keyword arguments, so `model(**{"input_features": x, "labels": y})` means `model(input_features=x, labels=y)`. It returns a `Seq2SeqLMOutput`; read `.loss` and `.logits`.
- **`with torch.no_grad():`** turns off gradient tracking for the indented block. Use it whenever you're only measuring, not training: it's faster and saves memory.
- **`F.cross_entropy(scores, targets, ignore_index=-100)`** (`F` is `torch.nn.functional`) computes the average cross-entropy. It expects `scores` as a 2-D tensor of shape `(N, 51864)`, one row per position, and `targets` as a 1-D tensor of shape `(N,)`.
- **`tensor.reshape(...)`** changes a tensor's shape without changing its data, as long as the total number of values stays the same. `logits.reshape(-1, 51864)` turns `(B, L, 51864)` into `(B × L, 51864)`; `-1` means "work this dimension out yourself". `labels.reshape(-1)` turns `(B, L)` into `(B × L,)`.

### TODO(you): compute the loss two ways

1. Collate the first 4 training examples into `batch`.
2. Run the model on it inside `torch.no_grad()`, and keep the result in `outputs`.
3. Compute the same loss yourself from `outputs.logits` and `batch["labels"]` with `F.cross_entropy`, into `my_loss`.

In [ ]:
# TODO(you): one forward pass, then the same loss by hand.
batch = ...
outputs = ...
my_loss = ...

print("logits shape:", tuple(outputs.logits.shape))
print(f"model's loss: {outputs.loss.item():.4f}   your loss: {my_loss.item():.4f}   "
      f"perplexity: {math.exp(outputs.loss.item()):.2f}")

In [ ]:
assert outputs.logits.shape == (4, batch["labels"].shape[1], model.config.vocab_size)
assert abs(outputs.loss.item() - my_loss.item()) < 1e-4, "your loss should match the model's"
print("Correct.")

### Which tokens are hard?

The loss is an average. Splitting it per token shows what fine-tuning will work on. `reduction="none"` makes `F.cross_entropy` return one loss per position instead of the average. Expect high losses on your jargon and names, and near-zero losses on common words.

The first row is the model predicting `<|notimestamps|>` right after `<|startoftranscript|>`. Its loss isn't zero, because Whisper was also trained to output timestamps, so on its own it isn't sure which mode you want. That position doesn't matter at inference, where `generate()` always supplies `<|notimestamps|>` as part of the prompt.

In [ ]:
with torch.no_grad():
    per_token = F.cross_entropy(outputs.logits[0], batch["labels"][0], ignore_index=-100, reduction="none")

tokens = processor.tokenizer.convert_ids_to_tokens([t for t in batch["labels"][0].tolist() if t != -100])
print(train_rows[0]["text"])
for token, loss in zip(tokens, per_token.tolist()):
    print(f"{token:>22}  {loss:6.2f}  {'#' * min(int(loss * 4), 40)}")

## Part 4: a sanity check, overfitting one batch

### One training step

Training repeats four lines:

    outputs = model(**batch)      # forward pass: predictions and loss
    outputs.loss.backward()       # backward pass: a gradient for every weight
    optimizer.step()              # update: move every weight a small step
    optimizer.zero_grad()         # clear the gradients for the next step

- **`loss.backward()`** runs **backpropagation**. For every weight, it computes the **gradient**: how much the loss would change if that weight increased a tiny bit. PyTorch stores it in the weight's `.grad` attribute, a tensor with the same shape as the weight. Whisper tiny has 37.8 million weights, so that's 37.8 million gradients per step.
- **`optimizer.step()`** moves each weight against its gradient, which lowers the loss. The **learning rate** sets how far. We use **AdamW**, the standard optimizer for transformers: it adapts the step size for each weight using the history of its gradients, and the "W" adds weight decay, which gently pulls weights toward zero.
- **`optimizer.zero_grad()`** clears the gradients. PyTorch **adds** each new gradient to whatever is already in `.grad`, so without this, every step would also apply all the previous steps' gradients again.
- **`model.train()`** puts the model in training mode, which turns on training-only behavior such as dropout. **`model.eval()`** turns it off for evaluation. Whisper tiny's dropout is set to 0, so here it changes nothing, but switch modes anyway: most models do use dropout.

`torch.optim.AdamW(model.parameters(), lr=1e-4)` creates the optimizer. `model.parameters()` hands it every weight tensor in the model, and `lr` is the learning rate.

### Why overfit one batch on purpose

Before spending 15 minutes on a real run, check that the training step works at all. Take one batch of 4 examples and train on it over and over. A working setup memorizes 4 sentences quickly, and the loss drops close to 0 within about 10 steps. If it doesn't, something is broken: wrong labels, a missing `zero_grad()` or `step()`, or a learning rate that's far off. This is one of the most useful debugging habits in ML.

The test wrecks the model, which memorizes 4 sentences and may now force them onto other audio, so the cell after it reloads a fresh copy. The learning rate here, 1e-4, is deliberately high so it memorizes fast. The real run uses 1e-5.

### TODO(you): overfit one batch

1. Call `model.train()`.
2. Create an AdamW optimizer with `lr=1e-4`.
3. Collate the first 4 training examples, once, into `batch`.
4. Repeat 10 times: forward pass, backward pass, optimizer step, zero the gradients, and append `outputs.loss.item()` to `losses`. Don't use `torch.no_grad()` here: the gradients are the whole point.

It takes about a minute, since every step does a forward and a backward pass.

In [ ]:
# TODO(you): train on one batch of 4 examples for 10 steps, recording the loss at each step.
losses = []
...

print([round(loss, 3) for loss in losses])

In [ ]:
assert len(losses) == 10, "record one loss per step"
assert all(isinstance(loss, float) for loss in losses), "use .item() to store plain floats"
assert losses[-1] < 0.25 * losses[0], "the loss should drop a lot when you train on the same batch over and over"

plt.figure(figsize=(6, 3))
plt.plot(losses, marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Overfitting one batch")
plt.show()

model.eval()
with torch.no_grad():
    memorized = model.generate(batch["input_features"][:1])
print("Label:    ", train_rows[0]["text"])
print("Generated:", processor.batch_decode(memorized, skip_special_tokens=True)[0].strip())
print("Correct.")

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
print("Reloaded a fresh whisper-tiny.en.")

## Part 5: the real training loop

### The words

- An **epoch** is one pass over every training example. With about 150 training recordings and batches of 8, an epoch is about 19 steps.
- **Shuffling** reorders the examples every epoch, so each batch is a different mix and the model can't learn anything from the order.
- The **learning rate** is 1e-5 (0.00001), ten times smaller than in Part 4. Small steps change the model gently, which matters with so little data: big steps would memorize your sentences and could make the model worse at everything else. Notebook 04 measures that and tries other values.
- **5 epochs.** The training loss should fall every epoch. The validation WER should fall at first and then level off. If it starts rising while the training loss keeps falling, the model is **overfitting**: memorizing the training sentences instead of learning something that carries over to new ones.

### The DataLoader

    loader = DataLoader(train_examples, batch_size=8, shuffle=True, collate_fn=collate)

- `train_examples` is your list of example dicts. A `DataLoader` accepts anything with a length and numbered items, and a list has both.
- `batch_size=8` puts 8 examples in each batch. The last batch is smaller if the count doesn't divide evenly.
- `shuffle=True` reshuffles at the start of every epoch.
- `collate_fn=collate` is your collator: the loader picks 8 examples and calls `collate` on that list.
- `for batch in loader:` gives you one batch per loop iteration, which is exactly the dict your `collate` returns. `len(loader)` is the number of batches per epoch.

### Validation WER during training

After each epoch, switch to `model.eval()`, transcribe the validation recordings with your `transcribe` from notebook 02 (now imported from `whisper_ft.py`), and compute `corpus_wer`. The next epoch switches back to `model.train()` inside `train_one_epoch`.

### TODO(you): write `train_one_epoch(model, loader, optimizer)`

**Inputs:** the model, a `DataLoader`, and an optimizer.

**Output:** the average training loss over the epoch, as a Python `float`.

**Steps:**
1. `model.train()`.
2. For each batch in the loader: forward pass, backward pass, optimizer step, and zero the gradients. Collect `outputs.loss.item()` in a list.
3. Return the mean of that list: `sum(values) / len(values)`.

It's your loop from Part 4, but over the loader's batches instead of one batch repeated.

In [ ]:
# TODO(you): one pass over the loader; return the average loss.
def train_one_epoch(model, loader, optimizer):
    ...

The check trains a copy of the model (`copy.deepcopy` copies every weight, so your model stays untouched) on 2 small batches, and confirms that the weights changed.

In [ ]:
check_model = copy.deepcopy(model)
check_loader = DataLoader(train_examples[:4], batch_size=2, collate_fn=collate)
check_optimizer = torch.optim.AdamW(check_model.parameters(), lr=1e-5)
weight_before = check_model.model.decoder.layers[0].fc1.weight.detach().clone()

loss = train_one_epoch(check_model, check_loader, check_optimizer)

assert isinstance(loss, float), f"return a Python float (use .item()), not {type(loss).__name__}"
assert 0 < loss < 20
assert not torch.equal(weight_before, check_model.model.decoder.layers[0].fc1.weight), \
    "the weights didn't change: is optimizer.step() called?"
assert check_model.training, "train_one_epoch should put the model in training mode"
del check_model, check_optimizer
print(f"Correct. Average loss on 2 small batches: {loss:.3f}")

### TODO(you): train for 5 epochs

1. Create `loader` as described above, and an AdamW optimizer with `lr=LEARNING_RATE`.
2. Before training, call `model.eval()`, measure the validation WER once, and append `{"epoch": 0, "train_loss": None, "val_wer": ...}` to `history`. That's your starting point. (`None` is Python's "no value".)
3. For each epoch from 1 to `EPOCHS` (`range(1, EPOCHS + 1)`): call `train_one_epoch`, switch to `model.eval()`, compute the validation WER, append `{"epoch": ..., "train_loss": ..., "val_wer": ...}` to `history`, and print a line so you can watch the progress.

The validation references and audio are ready in `val_refs` and `val_audios`. The whole run takes about 15 minutes. **Plug in your laptop first:** on battery, Windows slows the processor down to save power, and the run can take several times longer.

In [ ]:
LEARNING_RATE = 1e-5
EPOCHS = 1 if SMOKE_TEST else 5
BATCH_SIZE = 8

val_refs = [row["text"] for row in val_rows]
val_audios = [read_wav(row["path"]) for row in val_rows]

# TODO(you): the training loop. Fill `history` with one dict per epoch (epoch 0 = before training).
history = []
...

In [ ]:
assert len(history) == EPOCHS + 1 and history[0]["epoch"] == 0, "expected one entry before training, plus one per epoch"
assert all(set(h) == {"epoch", "train_loss", "val_wer"} for h in history)
if EPOCHS > 1:
    assert history[-1]["train_loss"] < history[1]["train_loss"], "the training loss should go down over the epochs"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot([h["epoch"] for h in history[1:]], [h["train_loss"] for h in history[1:]], marker="o")
ax1.set_xlabel("epoch")
ax1.set_title("Training loss")
ax2.plot([h["epoch"] for h in history], [h["val_wer"] for h in history], marker="o", color="tab:orange")
ax2.set_xlabel("epoch (0 = before training)")
ax2.set_title("Validation WER")
plt.tight_layout()
plt.show()
print("Correct.")

### Reading the curves

- **Training loss going down** means the model fits the training sentences better. It almost always goes down, so on its own it tells you little.
- **Validation WER going down** is what you actually want: better transcripts of sentences the model never trained on.
- **Validation WER flat or rising while the training loss falls** is overfitting. The epoch with the lowest validation WER is the one to keep; notebook 04 handles this properly.
- Your validation set is small (about 20 sentences, roughly 200 words), so each word is worth about 0.5 percentage points. Don't read much into small wiggles.

## Part 6: what changed?

Compare the transcripts before and after fine-tuning, on validation sentences only; the test set stays locked until notebook 04. A fresh whisper-tiny.en gives the "before" transcripts. The list comprehension keeps only the sentences whose transcript changed.

In [ ]:
fresh = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).eval()
before = transcribe(fresh, val_audios)
model.eval()
after = transcribe(model, val_audios)
del fresh

changed = [(ref, b, a) for ref, b, a in zip(val_refs, before, after) if b.strip() != a.strip()]
print(f"{len(changed)} of {len(val_refs)} validation transcripts changed. "
      f"WER {corpus_wer(val_refs, before):.1%} -> {corpus_wer(val_refs, after):.1%}\n")
for ref, b, a in changed[:10]:
    print(f"reference: {ref}\nbefore:    {b.strip()}\nafter:     {a.strip()}\n")

## Part 7: save the model

`model.save_pretrained(folder)` writes everything needed to load the model again:

- `config.json`: the architecture (layer sizes, vocabulary size, special token ids).
- `generation_config.json`: the settings `generate()` uses, like the prompt.
- `model.safetensors`: the 37.8 million weights, about 150 MB. "safetensors" is a file format that stores tensors safely and loads them quickly.

`processor.save_pretrained(folder)` adds the tokenizer and feature extractor settings, so the folder is complete on its own, and `WhisperForConditionalGeneration.from_pretrained(folder)` loads it back. The `models/` folder is in `.gitignore`: the weights are too big for GitHub.

`folder.iterdir()` lists the files in a folder, and `path.stat().st_size` is a file's size in bytes.

In [ ]:
out_dir = MODELS_DIR / "tiny-en-ft"
model.save_pretrained(out_dir)
processor.save_pretrained(out_dir)
save_json({"learning_rate": LEARNING_RATE, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "history": history},
          RESULTS_DIR / "notebook03_history.json")

for path in sorted(out_dir.iterdir()):
    print(f"{path.name:28} {path.stat().st_size / 1e6:7.1f} MB")

reloaded = WhisperForConditionalGeneration.from_pretrained(out_dir).eval()
print("\nThe reloaded model transcribes the first validation recording as:", transcribe(reloaded, val_audios[:1])[0].strip())

## Part 8: move your functions into `whisper_ft.py`

Notebook 04 trains several models, so it needs `prepare_example`, `collate`, and `train_one_epoch`. As in notebook 02, paste them into `whisper_ft.py`, replacing the three stubs under "From notebook 03".

They can use `processor`, `read_wav`, `SAMPLE_RATE`, `SOT`, `NO_TIMESTAMPS`, `EOT`, `torch`, and `np` there, since `whisper_ft.py` defines or imports all of them. It has no `model`, though: if one of your functions uses `model.config...`, change it to use one of those names instead.

In [ ]:
import whisper_ft

assert "NotImplementedError" not in inspect.getsource(whisper_ft.train_one_epoch), "paste your train_one_epoch too"
assert whisper_ft.prepare_example(train_rows[0])["labels"] == prepare_example(train_rows[0])["labels"]
module_batch = whisper_ft.collate(train_examples[:3])
notebook_batch = collate(train_examples[:3])
assert torch.equal(module_batch["labels"], notebook_batch["labels"])
assert torch.equal(module_batch["input_features"], notebook_batch["input_features"])
print("Correct. Notebook 04 can import them now.")

## What's next

You fine-tuned Whisper with a training loop you wrote yourself. Notebook 04 asks whether it really helped: WER on the locked test set, a check that general speech didn't get worse, and experiments with the learning rate, a frozen encoder, and LoRA.

Commit your work (`models/` stays out of git):

    git add -A
    git commit -m "Notebook 03: fine-tuning"
    git push

### Check your understanding

1. Why does the model shift the labels one position to the right to build the decoder input? What is this called, and how is it different from what `generate()` does?
2. Why pad labels with -100? Why can't you find the padding by looking for the `<|endoftext|>` id?
3. What did the decoder-input check catch, and what would the model have learned if you hadn't fixed it?
4. What did overfitting one batch prove? If the loss had stayed flat, what would you check first?
5. What happens if you forget `optimizer.zero_grad()`?
6. Your training loss went down every epoch. Does that mean the model got better at your voice? What tells you?
7. If 10,000 Flow users each had their own fully fine-tuned copy of Whisper, what would that cost to store and serve? What could you do instead? (Notebook 04 tries one answer.)